# ClinicalRAG — RAGAS Evaluation Notebook
**Project**: big_data_team1 · MIMIC-IV Clinical Notes RAG  
**Purpose**: Systematically evaluate RAG pipeline quality using the RAGAS framework

---

## Architecture Overview

| Mode | Method | Use Case | Available Metrics |
|------|--------|----------|-------------------|
| **Mode 1** | Hand-crafted Q&A dataset (`RAG_evaluation_dataset.xlsx`) | Known ground truth, precise evaluation | All 4 |
| **Mode 2** | Auto-generate questions from `discharge.csv`, then evaluate | Expand test coverage beyond the dataset | All 4 |

**RAGAS Four Metrics**:

| Metric | What it measures | Requires ground_truth? |
|--------|-----------------|------------------------|
| **Faithfulness** | Are answer claims supported by retrieved context? (hallucination detection) | No |
| **Answer Relevancy** | Does the answer actually address the question? | No |
| **Context Precision** | What fraction of retrieved docs are truly relevant to the ground truth? | Yes |
| **Context Recall** | How much of the information needed by ground truth is covered by retrieved docs? | Yes |

**Setup**: Place this notebook in the project root (same level as `app.py`) and ensure the FAISS index exists.

```
big_data_team1-main/
├── app.py
├── ragas_evaluation_en.ipynb   ← this file
├── RAG_evaluation_dataset.xlsx
├── discharge.csv               ← Mode 2 raw source
├── src/
└── storage/faiss_index/        ← build first with src/ingest.py
```


## Cell 1 — Install Dependencies

Install required packages. `ragas==0.2.9` is the tested stable version — do not upgrade arbitrarily as the API may change.

In [ ]:
# Run once to install required packages. Skip if already installed.
import subprocess, sys
packages = [
    "ragas==0.2.9", "datasets", "openai", "openpyxl", "pandas",
    "langchain-openai", "langchain-community", "matplotlib", "seaborn", "faiss-cpu",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("All packages installed.")


## Cell 2 — Global Imports

Import all required libraries and suppress unnecessary warnings to keep output clean.

In [ ]:
import os, sys, json, time, warnings
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")

# Make src/ importable from the project root
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
load_dotenv()

# RAGAS
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator

# LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document

print("Imports complete.")


## Cell 3 — Configuration (**only cell you need to modify**)

- `OPENAI_API_KEY`: your OpenAI API key (or set `OPENAI_API_KEY=sk-...` in a `.env` file)
- `DISCHARGE_CSV_PATH`: path to the raw MIMIC-IV discharge notes CSV for Mode 2
- `MODE2_NROWS`: how many notes to read from CSV (more = more stable generation, higher token cost)
- `MODE2_TESTSET_SIZE`: number of Q&A pairs to auto-generate (5~15 recommended)
- `SLEEP_BETWEEN_CALLS`: seconds between API calls to avoid rate limiting


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  USER CONFIG — Only modify this cell
# ─────────────────────────────────────────────────────────────────

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_API_KEY_HERE")
CHAT_MODEL     = os.getenv("OPENAI_CHAT_MODEL",  "gpt-4.1-mini")
EMBED_MODEL    = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

# Mode 1 — Q&A dataset
EVAL_DATASET_PATH = "./RAG_evaluation_dataset.xlsx"
QA_SHEET_NAME     = "RAG_evaluation_dataset_new"

# Mode 2 — raw discharge notes CSV (MIMIC-IV)
DISCHARGE_CSV_PATH = "./discharge.csv"   # <── set this to your actual CSV path
MODE2_NROWS        = 50                  # how many notes to load for generation
MODE2_TESTSET_SIZE = 8                   # auto-generated Q&A pairs (5~15 recommended)

OUTPUT_DIR            = "./evaluation_results"
SLEEP_BETWEEN_CALLS   = 1.5             # seconds between pipeline API calls

# ─────────────────────────────────────────────────────────────────
#  Validation
# ─────────────────────────────────────────────────────────────────
assert OPENAI_API_KEY and OPENAI_API_KEY != "YOUR_API_KEY_HERE", \
    "Set OPENAI_API_KEY in this cell or in a .env file."

from src.config import config
assert os.path.exists(config.FAISS_INDEX_DIR), (
    f"FAISS index not found at: {config.FAISS_INDEX_DIR}\n"
    "Run build_index() in src/ingest.py first."
)
assert os.path.exists(EVAL_DATASET_PATH), \
    f"Evaluation dataset not found: {EVAL_DATASET_PATH}"
assert os.path.exists(DISCHARGE_CSV_PATH), \
    f"discharge.csv not found at: {DISCHARGE_CSV_PATH}\n" \
    "Set DISCHARGE_CSV_PATH to your MIMIC-IV discharge notes CSV."

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration OK")
print(f"  Chat model       : {CHAT_MODEL}")
print(f"  Embed model      : {EMBED_MODEL}")
print(f"  FAISS index      : {config.FAISS_INDEX_DIR}")
print(f"  Discharge CSV    : {DISCHARGE_CSV_PATH}")
print(f"  Output dir       : {OUTPUT_DIR}")


## Cell 4 — Build RAGAS LLM & Embeddings

Each RAGAS metric internally calls an LLM (as judge) and Embeddings (for semantic similarity).  
RAGAS requires LangChain models to be wrapped before passing in. Built once, shared by both modes.


In [ ]:
def build_ragas_llm_and_embeddings():
    """Wrap OpenAI models in RAGAS-compatible wrappers."""
    llm = LangchainLLMWrapper(
        ChatOpenAI(model=CHAT_MODEL, api_key=OPENAI_API_KEY, temperature=0)
    )
    embeddings = LangchainEmbeddingsWrapper(
        OpenAIEmbeddings(model=EMBED_MODEL, api_key=OPENAI_API_KEY)
    )
    return llm, embeddings


def inject_llm_into_metrics(llm, embeddings):
    """Assign LLM and Embeddings to each RAGAS metric before evaluate()."""
    for metric in [faithfulness, answer_relevancy, context_precision, context_recall]:
        metric.llm = llm
        if hasattr(metric, "embeddings"):
            metric.embeddings = embeddings


ragas_llm, ragas_embeddings = build_ragas_llm_and_embeddings()
inject_llm_into_metrics(ragas_llm, ragas_embeddings)
print("RAGAS LLM and Embeddings initialized.")


## Cell 5 — Shared Helper: Run RAG Pipeline on a Batch of Questions

This function sends a batch of questions through the RAG pipeline, collecting each question's **answer** and **retrieved documents** (`contexts`), formatted for RAGAS:

```
{question, answer, contexts, ground_truth}
```


In [ ]:
from src.pipeline import run_pipeline


def run_pipeline_batch(questions: list, ground_truths: list) -> list:
    """
    Execute the full RAG pipeline for each question and collect
    RAGAS-formatted records.

    Parameters
    ----------
    questions     : list of question strings
    ground_truths : list of reference answer strings (same length)

    Returns
    -------
    List of dicts: {question, answer, contexts, ground_truth}
    """
    assert len(questions) == len(ground_truths), \
        "questions and ground_truths must have equal length."

    records = []
    total   = len(questions)

    for i, (q, gt) in enumerate(zip(questions, ground_truths), start=1):
        preview = q[:85] + "..." if len(q) > 85 else q
        print(f"  [{i:02d}/{total}] {preview}")

        try:
            result = run_pipeline(q)

            answer = result.answer or "Unable to confirm from the available retrieved notes."

            # Prefer reranked docs; fall back to retrieved docs
            docs_source = result.reranked_docs or result.retrieved_docs
            contexts = [
                d.get("content", "").strip()
                for d in docs_source
                if d.get("content", "").strip()
            ]
            if not contexts:
                contexts = ["[No context retrieved by the pipeline]"]

            print(f"        answer={len(answer)} chars | contexts={len(contexts)} chunks")

        except Exception as e:
            print(f"        Pipeline error: {e}")
            answer   = "[Pipeline error]"
            contexts = ["[Pipeline error — no context retrieved]"]

        records.append({
            "question":     q,
            "answer":       answer,
            "contexts":     contexts,
            "ground_truth": gt,
        })

        time.sleep(SLEEP_BETWEEN_CALLS)

    return records


print("Pipeline batch runner defined.")


## Cell 6 — Shared Helper: Run RAGAS Evaluation

Accepts pipeline output records, calls RAGAS to compute all four metrics, and prints a visual progress bar summary.


In [ ]:
def run_ragas(records: list, label: str = ""):
    """
    Run RAGAS evaluation on pipeline output records.

    Parameters
    ----------
    records : output of run_pipeline_batch()
    label   : display label for console output

    Returns
    -------
    (ragas_result, detail_df, scores_dict)
    """
    dataset = Dataset.from_list([
        {
            "question":     r["question"],
            "answer":       r["answer"],
            "contexts":     r["contexts"],
            "ground_truth": r["ground_truth"],
        }
        for r in records
    ])

    print(f"\nRunning RAGAS [{label}] on {len(records)} samples...")

    ragas_result = evaluate(
        dataset=dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    )

    detail_df = ragas_result.to_pandas()
    scores = {
        "faithfulness":      round(ragas_result["faithfulness"],      4),
        "answer_relevancy":  round(ragas_result["answer_relevancy"],  4),
        "context_precision": round(ragas_result["context_precision"], 4),
        "context_recall":    round(ragas_result["context_recall"],    4),
    }

    print(f"\n{'─'*55}")
    print(f"  RAGAS Results — {label}")
    print(f"{'─'*55}")
    for k, v in scores.items():
        bar = chr(9608) * int(v * 25)
        print(f"  {k:<22} {v:.4f}  |{bar:<25}|")
    print(f"{'─'*55}")

    return ragas_result, detail_df, scores


print("RAGAS evaluation helper defined.")


---
# ══════════════════════════════════════════
# MODE 1 — Evaluate with Your Q&A Dataset
# ══════════════════════════════════════════

**Full logic**:
1. Load Q&A pairs (`question` + `ground_truth`) from `RAG_evaluation_dataset.xlsx`
2. Run each question through the RAG pipeline to get system answers and retrieved contexts
3. Send `{question, answer, contexts, ground_truth}` to RAGAS for GPT-judged scoring
4. Save aggregate JSON summary and per-question Excel detail

**Advantage**: Ground truth is human-verified — most reliable evaluation.  
**Limitation**: Only evaluates questions in the dataset; does not reflect performance on arbitrary inputs.

## Cell 7 — Mode 1: Load Q&A Dataset


In [ ]:
# Load Mode 1 Q&A pairs from the hand-crafted evaluation dataset
df_qa = pd.read_excel(EVAL_DATASET_PATH, sheet_name=QA_SHEET_NAME)

required_cols = {"question", "ground_truth"}
assert not (required_cols - set(df_qa.columns)), \
    f"Missing columns: {required_cols - set(df_qa.columns)}"

df_qa = df_qa.dropna(subset=["question", "ground_truth"]).reset_index(drop=True)
df_qa["question"]     = df_qa["question"].astype(str).str.strip()
df_qa["ground_truth"] = df_qa["ground_truth"].astype(str).str.strip()

print(f"Loaded {len(df_qa)} Q&A pairs from Mode 1 dataset.")
df_qa[["id", "question", "ground_truth"]].head(5)


## Cell 8 — Mode 1: Run RAG Pipeline

Runs the pipeline question by question. Results are automatically cached to JSON so RAGAS can be re-run without repeating pipeline API calls.


In [ ]:
print("Mode 1 — Running RAG pipeline on Q&A dataset...\n")

m1_records = run_pipeline_batch(
    questions=df_qa["question"].tolist(),
    ground_truths=df_qa["ground_truth"].tolist(),
)

# Cache pipeline outputs so RAGAS can be re-run without repeating API calls
m1_cache_path = os.path.join(OUTPUT_DIR, "mode1_pipeline_cache.json")
with open(m1_cache_path, "w", encoding="utf-8") as f:
    json.dump(m1_records, f, indent=2, ensure_ascii=False)

print(f"\nPipeline outputs cached to: {m1_cache_path}")
print("(If RAGAS fails later, reload from cache in the optional cell below.)")


## Cell 8b — (Optional) Reload Mode 1 Pipeline Cache

If the pipeline has already been run, uncomment this cell to skip re-running it.

In [ ]:
# OPTIONAL — Uncomment to reload Mode 1 pipeline cache (skip re-running the pipeline)
# m1_cache_path = os.path.join(OUTPUT_DIR, "mode1_pipeline_cache.json")
# with open(m1_cache_path, encoding="utf-8") as f:
#     m1_records = json.load(f)
# print(f"Reloaded {len(m1_records)} Mode 1 records from cache.")


## Cell 9 — Mode 1: RAGAS Evaluation

RAGAS calls GPT internally to judge each question using LLM-as-judge. Expect 2–5 minutes.

In [ ]:
m1_ragas_result, m1_detail_df, m1_scores = run_ragas(
    m1_records, label="Mode 1 — Q&A Dataset"
)


## Cell 10 — Mode 1: Save & Inspect Results

Saves a JSON aggregate summary and an Excel per-question detail file, then renders a colour-coded heatmap to identify low-scoring questions.


In [ ]:
# Save summary JSON
m1_summary_path = os.path.join(OUTPUT_DIR, "mode1_summary.json")
with open(m1_summary_path, "w", encoding="utf-8") as f:
    json.dump(m1_scores, f, indent=2)

# Merge original meta columns for traceability
meta_cols = [c for c in ["id", "use_case", "note_id_reference"] if c in df_qa.columns]
m1_detail_enriched = pd.concat(
    [df_qa[meta_cols].reset_index(drop=True), m1_detail_df.reset_index(drop=True)],
    axis=1,
)
m1_detail_path = os.path.join(OUTPUT_DIR, "mode1_detail.xlsx")
m1_detail_enriched.to_excel(m1_detail_path, index=False)

print(f"Mode 1 results saved:")
print(f"  Summary -> {m1_summary_path}")
print(f"  Detail  -> {m1_detail_path}\n")

# Heatmap table
score_cols = [c for c in ["faithfulness","answer_relevancy","context_precision","context_recall"]
              if c in m1_detail_enriched.columns]
display(m1_detail_enriched[["id", "question"] + score_cols].style.background_gradient(
    subset=score_cols, cmap="RdYlGn", vmin=0, vmax=1
))


---
# ══════════════════════════════════════════════════════
# MODE 2 — Auto-Generate Test Set from discharge.csv
# ══════════════════════════════════════════════════════

**Problem solved**: The Q&A dataset has limited questions; Mode 2 evaluates system performance on arbitrary new inputs.

**Full logic**:
1. Load raw MIMIC-IV discharge notes from `discharge.csv`
2. Convert to LangChain `Document` objects and pass to **RAGAS `TestsetGenerator`**
3. `TestsetGenerator` calls GPT to **automatically read documents and generate N {question, ground_truth} pairs**
4. Send auto-generated questions through the RAG pipeline
5. Pipeline output + auto-generated ground truth → RAGAS scoring

**TestsetGenerator output columns**:
- `user_input`: auto-generated question
- `reference`: auto-generated reference answer (grounded in source text)
- `reference_contexts`: document chunks used during generation

> ⚠️ Auto-generated ground truth quality depends on GPT. Manually review 3–5 pairs to confirm they are clinically reasonable.

## Cell 11 — Mode 2: Load Clinical Notes from discharge.csv


In [ ]:
# ── Load raw MIMIC-IV discharge notes from CSV ────────────────────────────────
# Supported columns: 'text' (required), 'note_id' (optional)
# discharge.csv typically contains columns: subject_id, hadm_id, note_id, charttime, text, ...

df_notes = pd.read_csv(
    DISCHARGE_CSV_PATH,
    nrows=MODE2_NROWS,
    usecols=lambda c: c in {"note_id", "subject_id", "hadm_id", "text"},
)

# Identify the text column (handle alternate naming)
text_col = "text" if "text" in df_notes.columns else df_notes.columns[-1]
df_notes = df_notes.dropna(subset=[text_col]).reset_index(drop=True)

raw_documents = [
    Document(
        page_content=str(row[text_col]).strip(),
        metadata={
            "note_id":    str(row.get("note_id", f"note_{i}")),
            "subject_id": str(row.get("subject_id", "")),
            "hadm_id":    str(row.get("hadm_id", "")),
            "source":     "discharge.csv",
        },
    )
    for i, row in df_notes.iterrows()
    if len(str(row[text_col]).strip()) > 50  # skip near-empty notes
]

print(f"Loaded {len(raw_documents)} discharge notes from CSV.")
avg_len = sum(len(d.page_content) for d in raw_documents) // max(len(raw_documents), 1)
print(f"  Avg note length : {avg_len} chars")
print(f"  Sample note IDs : {[d.metadata['note_id'] for d in raw_documents[:5]]}")


## Cell 12 — Mode 2: Auto-Generate Test Set with RAGAS TestsetGenerator

`TestsetGenerator` workflow:
1. Reads all documents, uses LLM to extract "knowledge nodes"
2. Designs questions targeting each node
3. Uses LLM to answer from source text, producing ground_truth
4. Returns N `{user_input, reference}` pairs

With `MODE2_TESTSET_SIZE=8` expect 2–5 minutes and additional OpenAI token usage.


In [ ]:
print(f"Initializing TestsetGenerator (will generate {MODE2_TESTSET_SIZE} test cases)...")
print("This calls OpenAI multiple times and may take 2-5 minutes.\n")

generator = TestsetGenerator(
    llm=ragas_llm,
    embedding_model=ragas_embeddings,
)

generated_testset = generator.generate_with_langchain_docs(
    documents=raw_documents,
    testset_size=MODE2_TESTSET_SIZE,
)

df_generated = generated_testset.to_pandas()

# Normalize column names across RAGAS versions
q_col  = "user_input" if "user_input" in df_generated.columns else "question"
gt_col = "reference"  if "reference"  in df_generated.columns else "ground_truth"

print(f"Generated {len(df_generated)} test cases.")
print(f"Columns: {list(df_generated.columns)}\n")
df_generated[[q_col, gt_col]].head(5)


## Cell 13 — Mode 2: Inspect & Save Generated Test Set

**Recommended**: manually review each Q&A pair. Unreasonable ones can be filtered before running the pipeline.


In [ ]:
# Save generated testset for audit
m2_testset_path = os.path.join(OUTPUT_DIR, "mode2_generated_testset.xlsx")
df_generated.to_excel(m2_testset_path, index=False)
print(f"Generated testset saved to: {m2_testset_path}")
print("Review quality — check that each Q&A pair is clinically reasonable.\n")

print("─" * 80)
for i, row in df_generated.iterrows():
    q  = str(row.get(q_col,  ""))
    gt = str(row.get(gt_col, ""))
    print(f"  Q{i+1}: {q[:110]}")
    print(f"  A{i+1}: {gt[:130]}")
    print()


## Cell 14 — Mode 2: Run RAG Pipeline on Generated Questions

Sends auto-generated questions through the RAG pipeline. Results are cached to JSON.


In [ ]:
m2_questions     = df_generated[q_col].dropna().astype(str).str.strip().tolist()
m2_ground_truths = df_generated[gt_col].dropna().astype(str).str.strip().tolist()

# Align lengths (guard against ragas returning fewer rows than requested)
min_len = min(len(m2_questions), len(m2_ground_truths))
m2_questions, m2_ground_truths = m2_questions[:min_len], m2_ground_truths[:min_len]

print(f"Mode 2 — Running RAG pipeline on {min_len} auto-generated questions...\n")

m2_records = run_pipeline_batch(
    questions=m2_questions,
    ground_truths=m2_ground_truths,
)

m2_cache_path = os.path.join(OUTPUT_DIR, "mode2_pipeline_cache.json")
with open(m2_cache_path, "w", encoding="utf-8") as f:
    json.dump(m2_records, f, indent=2, ensure_ascii=False)
print(f"\nMode 2 pipeline outputs cached to: {m2_cache_path}")


## Cell 14b — (Optional) Reload Mode 2 Pipeline Cache

If the pipeline has already been run, uncomment this cell to skip re-running it.

In [ ]:
# OPTIONAL — Uncomment to reload Mode 2 pipeline cache
# m2_cache_path = os.path.join(OUTPUT_DIR, "mode2_pipeline_cache.json")
# with open(m2_cache_path, encoding="utf-8") as f:
#     m2_records = json.load(f)
# print(f"Reloaded {len(m2_records)} Mode 2 records from cache.")


## Cell 15 — Mode 2: RAGAS Evaluation

In [ ]:
m2_ragas_result, m2_detail_df, m2_scores = run_ragas(
    m2_records, label="Mode 2 — Auto-Generated"
)


## Cell 16 — Mode 2: Save Results

In [ ]:
m2_summary_path = os.path.join(OUTPUT_DIR, "mode2_summary.json")
with open(m2_summary_path, "w", encoding="utf-8") as f:
    json.dump(m2_scores, f, indent=2)

m2_detail_path = os.path.join(OUTPUT_DIR, "mode2_detail.xlsx")
m2_detail_df.to_excel(m2_detail_path, index=False)

print(f"Mode 2 results saved:")
print(f"  Summary -> {m2_summary_path}")
print(f"  Detail  -> {m2_detail_path}\n")

score_cols_m2 = [c for c in ["faithfulness","answer_relevancy","context_precision","context_recall"]
                 if c in m2_detail_df.columns]
display(m2_detail_df[["question"] + score_cols_m2].style.background_gradient(
    subset=score_cols_m2, cmap="RdYlGn", vmin=0, vmax=1
))


---
# ═══════════════════════════════════════
# COMBINED ANALYSIS — Compare Both Modes
# ═══════════════════════════════════════

**Result interpretation**:

| Situation | Diagnosis |
|-----------|-----------|
| Both modes score similarly | Good generalisation — system handles novel questions well |
| Mode 2 significantly lower | System may be overfitted to the Q&A dataset |
| Low `Faithfulness` | LLM hallucination — strengthen prompt or improve retrieval |
| Low `Answer Relevancy` | LLM is off-topic — check query rewriting or generation prompt |
| Low `Context Precision` | Too many irrelevant documents retrieved — tune TOP_K or embeddings |
| Low `Context Recall` | Required documents not retrieved — increase TOP_K or adjust chunk size |

## Cell 17 — Combined Summary Table


In [ ]:
metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

summary_df = pd.DataFrame({
    "Metric":                  metrics,
    "Mode 1 (Q&A Dataset)":    [m1_scores[m] for m in metrics],
    "Mode 2 (Auto-Generated)": [m2_scores[m] for m in metrics],
})
summary_df["Delta (M2 - M1)"] = (
    summary_df["Mode 2 (Auto-Generated)"] - summary_df["Mode 1 (Q&A Dataset)"]
).round(4)

combined_path = os.path.join(OUTPUT_DIR, "combined_summary.xlsx")
summary_df.to_excel(combined_path, index=False)
print(f"Combined summary saved to: {combined_path}\n")

display(
    summary_df.style
    .background_gradient(
        subset=["Mode 1 (Q&A Dataset)", "Mode 2 (Auto-Generated)"],
        cmap="RdYlGn", vmin=0, vmax=1,
    )
    .applymap(
        lambda v: "color: green; font-weight: bold" if v > 0.02
                  else "color: red; font-weight: bold" if v < -0.02
                  else "",
        subset=["Delta (M2 - M1)"],
    )
)


## Cell 18 — Radar Chart Comparison

Visualises all four metrics simultaneously. Suitable for reports and presentations.

In [ ]:
labels  = ["Faithfulness", "Answer\nRelevancy", "Context\nPrecision", "Context\nRecall"]
m1_vals = [m1_scores[m] for m in metrics]
m2_vals = [m2_scores[m] for m in metrics]

N      = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"], fontsize=8, color="gray")

for vals, color, marker, label in [
    (m1_vals, "#2196F3", "o", "Mode 1 — Q&A Dataset"),
    (m2_vals, "#FF5722", "s", "Mode 2 — Auto-Generated"),
]:
    v_plot = vals + vals[:1]
    ax.plot(angles, v_plot, f"{marker}-", linewidth=2, color=color, label=label)
    ax.fill(angles, v_plot, alpha=0.12, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11)
ax.set_title("ClinicalRAG — RAGAS Evaluation\nMode 1 vs Mode 2",
             fontsize=13, pad=22, fontweight="bold")
ax.legend(loc="upper right", bbox_to_anchor=(1.38, 1.15), fontsize=10)

radar_path = os.path.join(OUTPUT_DIR, "ragas_radar_chart.png")
plt.tight_layout()
plt.savefig(radar_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Radar chart saved to: {radar_path}")


## Cell 19 — Bar Chart Comparison

Shows exact metric values with score labels. Suitable for reports.

In [ ]:
x     = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

bars1 = ax.bar(x - width/2, [m1_scores[m] for m in metrics], width,
               label="Mode 1 — Q&A Dataset",    color="#42A5F5", edgecolor="white")
bars2 = ax.bar(x + width/2, [m2_scores[m] for m in metrics], width,
               label="Mode 2 — Auto-Generated", color="#FF7043", edgecolor="white")

for bar, color in [(bars1, "#1565C0"), (bars2, "#BF360C")]:
    for b in bar:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.012,
                f"{b.get_height():.3f}", ha="center", va="bottom",
                fontsize=9, color=color, fontweight="bold")

ax.set_ylim(0, 1.18)
ax.set_xticks(x)
ax.set_xticklabels(["Faithfulness","Answer Relevancy","Context Precision","Context Recall"],
                    fontsize=11)
ax.set_ylabel("Score (0–1)", fontsize=11)
ax.set_title("ClinicalRAG — RAGAS Metric Comparison", fontsize=13, fontweight="bold", pad=12)
ax.axhline(y=0.7, color="gray", linestyle="--", linewidth=0.8, alpha=0.5,
           label="0.7 reference line")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
sns.despine()

bar_path = os.path.join(OUTPUT_DIR, "ragas_bar_chart.png")
plt.tight_layout()
plt.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Bar chart saved to: {bar_path}")


## Cell 20 — Mode 1 Per-Question Score Distribution (Box Plot)

Outliers indicate questions where the system underperforms — useful for targeted debugging.

In [ ]:
score_cols = [c for c in ["faithfulness","answer_relevancy","context_precision","context_recall"]
              if c in m1_detail_df.columns]

fig, axes = plt.subplots(1, len(score_cols), figsize=(4*len(score_cols), 5), sharey=True)
palette   = ["#42A5F5", "#66BB6A", "#FFA726", "#EF5350"]

for ax, col, color in zip(axes, score_cols, palette):
    vals = m1_detail_df[col].dropna()
    ax.boxplot(vals, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color="black", linewidth=2))
    ax.scatter([1]*len(vals), vals, alpha=0.65, color=color, s=45, zorder=3)
    ax.set_title(col.replace("_","\n"), fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.set_xticks([])
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Mode 1 — Per-Question Score Distribution",
             fontsize=13, fontweight="bold", y=1.02)
dist_path = os.path.join(OUTPUT_DIR, "mode1_score_distribution.png")
plt.tight_layout()
plt.savefig(dist_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Distribution chart saved to: {dist_path}")


## Cell 21 — Final Output Checklist

Verifies that all output files were saved successfully.

In [ ]:
print("=" * 65)
print("  ClinicalRAG RAGAS Evaluation — Output Checklist")
print("=" * 65)

output_files = [
    ("Mode 1 Pipeline cache",       "mode1_pipeline_cache.json"),
    ("Mode 1 RAGAS summary",         "mode1_summary.json"),
    ("Mode 1 Per-question detail",   "mode1_detail.xlsx"),
    ("Mode 2 Generated testset",     "mode2_generated_testset.xlsx"),
    ("Mode 2 Pipeline cache",        "mode2_pipeline_cache.json"),
    ("Mode 2 RAGAS summary",         "mode2_summary.json"),
    ("Mode 2 Per-question detail",   "mode2_detail.xlsx"),
    ("Combined summary table",       "combined_summary.xlsx"),
    ("Radar chart",                  "ragas_radar_chart.png"),
    ("Bar chart",                    "ragas_bar_chart.png"),
    ("Score distribution (Mode 1)",  "mode1_score_distribution.png"),
]

for desc, fname in output_files:
    full_path = os.path.join(OUTPUT_DIR, fname)
    exists    = os.path.exists(full_path)
    status    = "OK     " if exists else "MISSING"
    print(f"  [{status}]  {desc}")
    if exists:
        print(f"             {full_path}")
    print()

print("=" * 65)
print("  Final Scores")
print("=" * 65)
print(f"  {'Metric':<24} {'Mode 1':>10} {'Mode 2':>10} {'Delta':>10}")
print("  " + "-" * 56)
for m in metrics:
    v1 = m1_scores[m]; v2 = m2_scores[m]; d = v2 - v1
    sign = "+" if d >= 0 else ""
    print(f"  {m:<24} {v1:>10.4f} {v2:>10.4f} {sign}{d:>9.4f}")
print("=" * 65)


---
## Appendix A — Existing Evaluator vs RAGAS

| Aspect | `src/evaluator.py` (existing) | RAGAS (this notebook) |
|--------|-------------------------------|-----------------------|
| Evaluation timing | During pipeline execution (internal gatekeeper) | After pipeline execution (external judge) |
| Decision logic | Rule-based: content length > 80 chars, rerank score ≥ 0.20 | LLM-as-judge: GPT verifies each claim |
| Hallucination detection | Not possible | Faithfulness metric |
| Requires ground truth | No | Yes (Mode 2 auto-generates it) |
| Compute cost | Very low (local) | Moderate (OpenAI API calls) |
| Recommended use | Real-time retrieval quality gate | Rigorous evaluation during development |

**Conclusion**: The two approaches are complementary — use both together for best results.

---
## Appendix B — Frequently Asked Questions

**Q1: Getting RateLimitError?**  
Increase `SLEEP_BETWEEN_CALLS` to 3–5 seconds.

**Q2: TestsetGenerator producing poor-quality questions?**  
This is normal. Manually filter out unreasonable pairs, or increase `MODE2_TESTSET_SIZE` for a more stable average.

**Q3: context_precision / context_recall showing NaN?**  
RAGAS cannot compute these when ground_truth is too short or contexts are empty. These samples are skipped in the average — this is expected behaviour.

**Q4: FAISS index does not exist?**  
Run `build_index()` in `src/ingest.py` first (one-time setup step).

**Q5: Want to run only one mode?**  
The two modes are fully independent. For Mode 1 only, run Cells 7–10. For Mode 2 only, run Cells 11–16.

**Q6: discharge.csv has different column names?**  
Cell 11 attempts automatic column detection. You can also manually set `text_col` to the correct column name.
